# Session 1: a building is a mass on a spring
**A. Real-world problem.** After an earthquake, some buildings are badly shaken and the
building next door is fine. Tall flexible buildings sway slowly, short stiff ones sway fast,
and the ground has its own favourite rhythm. When the two rhythms match, the sway grows. Today
you build the smallest model that shows this: one lump of mass on one spring.

**Plan.** (1) the sway period from mass and stiffness, (2) the sway as a wave in time,
(3) a rule of thumb for real buildings, (4) how much the building amplifies slow, matched
and fast shaking.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: the period of the sway
Picture the whole building as a mass m (kg) on a spring of stiffness k (newtons per metre of
sideways push). Heavier means slower; stiffer means faster. The time for one full back-and-forth
sway is the *period* T:

    w = sqrt(k / m)        (angular frequency, radians per second)
    T = 2 pi / w           (period, seconds)

In [ ]:
def sway_frequency(k, m):
    """Angular frequency w = sqrt(k / m) in rad/s (k in N/m, m in kg)."""
    return float(np.sqrt(k / m))


def sway_period(k, m):
    """Period T = 2 pi / w in seconds."""
    return float(2 * np.pi / sway_frequency(k, m))

## Step 2: the sway in time
Push the building and let go. It swings back and forth like a pendulum, and slowly dies away.
Here we only draw the clean wave: x(t) = x0 cos(w t).

In [ ]:
def free_sway(t, x0, k, m):
    """Sideways displacement of the roof in metres at time t (s), starting at rest from x0."""
    return x0 * np.cos(sway_frequency(k, m) * np.asarray(t, float))

## Step 3: a rule of thumb
Engineers often guess the period of a typical frame building (columns and beams) as
**T about 0.1 s per storey**. It is only a rough empirical guide: shear walls, soft ground
and tall slender towers can be very different.

In [ ]:
def rule_of_thumb_period(storeys):
    """T = 0.1 N seconds (rough rule for typical frame buildings)."""
    return 0.1 * storeys

## Step 4: resonance, first look
Shake the ground back and forth with period T_g. If the shaking is much slower than the
building's own period, the building just follows the ground (gain about 1). If T_g equals the
building's period, each push adds to the last, and a typical damped building sways about ten
times more. If the shaking is much faster, the building hardly moves. The *gain* is the sway
divided by the sway for a very slow push. Session 2 derives the formula; here we just use it.
With r = T / T_g (building period over ground period) and damping ratio zeta:

    gain = 1 / sqrt((1 - r^2)^2 + (2 zeta r)^2)

In [ ]:
def gain(t_building, t_ground, zeta=0.05):
    """Dynamic gain (no unit) of a damped building shaken harmonically with period t_ground."""
    r = np.asarray(t_building, float) / np.asarray(t_ground, float)
    return 1.0 / np.sqrt((1 - r ** 2) ** 2 + (2 * zeta * r) ** 2)

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    m, k = 2.0e5, 2.0e7
    print(f"m = {m:.0f} kg, k = {k:.0f} N/m -> w = {sway_frequency(k, m):.3f} rad/s, T = {sway_period(k, m):.4f} s")
    cases = [("reference", k, m), ("heavier (2 m)", k, 2 * m), ("stiffer (4 k)", 4 * k, m)]
    t = np.linspace(0, 4, 800)
    fig, ax = plt.subplots(figsize=(7, 3.6))
    for name, kk, mm in cases:
        T = sway_period(kk, mm)
        ax.plot(t, 100 * free_sway(t, 0.05, kk, mm), label=f"{name}: T = {T:.3f} s")
        print(f"{name}: T = {T:.4f} s")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("roof sway (cm)")
    ax.set_title("Heavy sways slowly, stiff sways fast")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "sway_periods.png", dpi=120)

    tg = np.linspace(0.2, 3.0, 400)
    fig2, ax2 = plt.subplots(figsize=(7, 3.6))
    for tb in (0.5, 1.0):
        ax2.plot(tg, gain(tb, tg), label=f"building period {tb} s")
    ax2.set_yscale("log")
    ax2.set_xlabel("ground shaking period (s)")
    ax2.set_ylabel("gain: sway / slow-push sway (no unit)")
    ax2.set_title("Resonance: the gain peaks when the periods match")
    ax2.grid(alpha=0.3, which="both")
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "gain_vs_ground_period.png", dpi=120)
    for n in (3, 10, 20):
        print(f"{n} storeys: rule of thumb T = {rule_of_thumb_period(n):.1f} s")
    print(f"gain when periods match (5% damping): {gain(1.0, 1.0):.2f}")

## Where this stops
A real building is not one lump: it bends in several patterns at once (Session 3), and the
gain above is for steady, repeated shaking, not for one real earthquake.

## Coding tasks
1. **Easy.** In `lesson.py`, print the period for m = 4e5 kg and k = 2e7 N/m. Predict it first from the worked example, then check.
2. **Stretch.** Write a small function that returns the ground period at which a building with a given mass and stiffness resonates, and test it on the worked example.
3. **Explain.** Explain, in two or three sentences and without formulas, why a 20-storey frame is more likely to be hurt by slow, long-period ground shaking than a 3-storey one. Use the rule of thumb.

## Assessment
- **A. Conceptual.** Why does a heavier building sway more slowly while a stiffer building sways faster? Why can a matching ground rhythm be dangerous even when the push itself is small?
- **B. Numerical.** A building lump has m = 500 000 kg and k = 20 000 000 N/m. Find w and T and say whether it is close to the 1 s ground period.
- **C. Graph and simulation interpretation.** On the gain graph a 1 s building is shaken by a ground whose period is 2 s, and then by one with period 0.5 s. Read the gain in each case and say what each tells an engineer.
- **D. Design and optimisation challenge.** A building has m = 200 000 kg and the ground period is 1 s. Choose a stiffness k that makes its period 0.5 s (a stiff design) or 1.5 s (a flexible design). What does each give for the gain, and what is the downside of each?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    m, k = 2.0e5, 2.0e7
    w, T = L.sway_frequency(k, m), L.sway_period(k, m)
    checks.append((f"worked example: m = 200000 kg, k = 20000000 N/m gives w = 10 rad/s and T = 0.6283 s (got {w:.3f}, {T:.4f})",
                   abs(w - 10.0) < 1e-9 and abs(T - 0.6283) < 5e-5))
    checks.append(("a period is 2 pi / w (T w = 2 pi)", abs(T * w - 2 * np.pi) < 1e-12))
    checks.append(("twice the mass: T grows by sqrt(2), 0.8886 s", abs(L.sway_period(k, 2 * m) - 0.8886) < 5e-5
                   and abs(L.sway_period(k, 2 * m) / T - np.sqrt(2)) < 1e-12))
    checks.append(("four times the stiffness halves the period, 0.3142 s", abs(L.sway_period(4 * k, m) - 0.3142) < 5e-5
                   and abs(L.sway_period(4 * k, m) / T - 0.5) < 1e-12))
    # free sway: starts at x0, comes back after exactly one period, is at -x0 after half a period
    x0 = 0.05
    checks.append(("free sway starts at x0 and returns after one period",
                   abs(L.free_sway(0.0, x0, k, m) - x0) < 1e-12 and abs(L.free_sway(T, x0, k, m) - x0) < 1e-12))
    checks.append(("free sway is -x0 after half a period", abs(L.free_sway(T / 2, x0, k, m) + x0) < 1e-12))
    # rule of thumb
    checks.append(("rule of thumb: 3 storeys 0.3 s, 10 storeys 1.0 s, 20 storeys 2.0 s",
                   abs(L.rule_of_thumb_period(3) - 0.3) < 1e-12 and abs(L.rule_of_thumb_period(10) - 1.0) < 1e-12
                   and abs(L.rule_of_thumb_period(20) - 2.0) < 1e-12))
    # resonance gain
    checks.append(("gain at matching periods with 5% damping is 10.00 (= 1 / (2 zeta))", abs(L.gain(1.0, 1.0) - 10.0) < 1e-9))
    checks.append(("very slow shaking: gain close to 1 (1.04 at r = 0.2)", abs(L.gain(0.2, 1.0) - 1.0414) < 5e-4))
    checks.append(("very fast shaking: gain well below 1 (0.33 at r = 2)", abs(L.gain(2.0, 1.0) - 0.3326) < 5e-4))
    checks.append(("gain is about 2 at r = 0.707 (1.98)", abs(L.gain(0.7071, 1.0) - 1.98) < 5e-3))
    # edge: more damping, smaller peak
    checks.append(("more damping lowers the resonance gain (zeta 0.1 gives 5.00)", abs(L.gain(1.0, 1.0, 0.1) - 5.0) < 1e-9))
    # edge: array input works
    g = L.gain(1.0, np.array([0.5, 1.0, 2.0]))
    checks.append(("gain accepts arrays, largest at matching period", g.shape == (3,) and int(np.argmax(g)) == 1))
    # assessment answers quoted in Section F
    checks.append(("B: m = 500000 kg, k = 20000000 N/m gives w = 6.325 rad/s and T = 0.9935 s",
                   abs(L.sway_frequency(2e7, 5e5) - 6.325) < 5e-4 and abs(L.sway_period(2e7, 5e5) - 0.9935) < 5e-5))
    k_half = 2e5 * (2 * np.pi / 0.5) ** 2
    k_long = 2e5 * (2 * np.pi / 1.5) ** 2
    checks.append(("D: T = 0.5 s needs k = 3.158e7 N/m (gain 1.33); T = 1.5 s needs k = 3.509e6 N/m (gain 0.79) for m = 200000 kg",
                   abs(k_half - 3.158e7) < 2e4 and abs(L.sway_period(k_half, 2e5) - 0.5) < 1e-12 and abs(L.gain(0.5, 1.0) - 1.33) < 5e-3
                   and abs(k_long - 3.509e6) < 2e3 and abs(L.sway_period(k_long, 2e5) - 1.5) < 1e-12 and abs(L.gain(1.5, 1.0) - 0.794) < 5e-4))
    checks.append(("C: a 1 s building shaken with a 2 s period has gain 1.33; with a 0.5 s period 0.33",
                   abs(L.gain(1.0, 2.0) - 1.33) < 5e-3 and abs(L.gain(1.0, 0.5) - 0.33) < 5e-3))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
